In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
from pathlib import Path
import pandas as pd
import re, pathlib

In [2]:
# =============================================================================
# 1.  Directory layout – pathlib all the way
# =============================================================================
SCRIPT_DIR   = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = SCRIPT_DIR.parent          # edit if your notebook is elsewhere

DATA_DIR       = PROJECT_ROOT / "data/"
SIMULATION_DIR = DATA_DIR / "simulations/"          # folder with ATTRIBUTE_* and wide SSP file
TORNADO_SIM_DIR = SIMULATION_DIR /"data_for_LSU"
OUTPUT_DIR     = DATA_DIR / "output/"

In [3]:
louisiana = pd.read_csv(TORNADO_SIM_DIR / "louisiana.csv")

In [ ]:
louisiana

In [4]:
import pandas as pd

# 1) Filter base case
base_case = louisiana[louisiana['primary_id'] == 0].copy()

# 2) List your transport fuels
relevant_fuels = [
    'diesel',
    'electricity',
    'gasoline',
    'hydrocarbon_gas_liquids',
    'hydrogen',
    'natural_gas'
]

# 3) Quick check of available demand & efficiency columns
print("=== available transport-demand columns ===")
for c in base_case.columns:
    if 'energy_demand_enfu_subsector_total_pj_trns_fuel_' in c:
        print(" ", c)
print("\n=== available transport-efficiency columns ===")
for c in base_case.columns:
    if 'fuelefficiency_trns_road_light' in c:
        print(" ", c)
print("\n")

=== available transport-demand columns ===
  energy_demand_enfu_subsector_total_pj_trns_fuel_ammonia
  energy_demand_enfu_subsector_total_pj_trns_fuel_biofuels
  energy_demand_enfu_subsector_total_pj_trns_fuel_biogas
  energy_demand_enfu_subsector_total_pj_trns_fuel_biomass
  energy_demand_enfu_subsector_total_pj_trns_fuel_coal
  energy_demand_enfu_subsector_total_pj_trns_fuel_coke
  energy_demand_enfu_subsector_total_pj_trns_fuel_crude
  energy_demand_enfu_subsector_total_pj_trns_fuel_diesel
  energy_demand_enfu_subsector_total_pj_trns_fuel_electricity
  energy_demand_enfu_subsector_total_pj_trns_fuel_furnace_gas
  energy_demand_enfu_subsector_total_pj_trns_fuel_gasoline
  energy_demand_enfu_subsector_total_pj_trns_fuel_geothermal
  energy_demand_enfu_subsector_total_pj_trns_fuel_hydrocarbon_gas_liquids
  energy_demand_enfu_subsector_total_pj_trns_fuel_hydrogen
  energy_demand_enfu_subsector_total_pj_trns_fuel_kerosene
  energy_demand_enfu_subsector_total_pj_trns_fuel_natural_gas
  en

In [5]:


# 4) Prepare accumulators
total_saved_volume = pd.Series(0.0, index=base_case.index)
demand_trns = pd.DataFrame({'time_period': base_case['time_period']},
                           index=base_case.index)

# 5) Loop & pattern-match per fuel
for fuel in relevant_fuels:
    # 5a) demand cols specific to this fuel
    dem_cols = [
        c for c in base_case.columns
        if f'energy_demand_enfu_subsector_total_pj_trns_fuel_{fuel}' in c
    ]
    # 5b) efficiency cols for this fuel
    eff_cols = [
        c for c in base_case.columns
        if f'fuelefficiency_trns_road_light_{fuel}' in c
    ]

    print(f"Fuel={fuel!r}: dem_cols={dem_cols}, eff_cols={eff_cols}")
    if not dem_cols or not eff_cols:
        print(f"  → skipping {fuel!r} (no matching columns)\n")
        continue

    fuel_demand     = base_case[dem_cols[0]]
    fuel_efficiency = base_case[eff_cols[0]]

    demand_trns[fuel] = fuel_demand

    vol_now  = fuel_demand / fuel_efficiency
    vol_base = fuel_demand / fuel_efficiency.iloc[0]
    total_saved_volume += (vol_base - vol_now)



Fuel='diesel': dem_cols=['energy_demand_enfu_subsector_total_pj_trns_fuel_diesel'], eff_cols=['fuelefficiency_trns_road_light_diesel_km_per_litre']
Fuel='electricity': dem_cols=['energy_demand_enfu_subsector_total_pj_trns_fuel_electricity'], eff_cols=[]
  → skipping 'electricity' (no matching columns)

Fuel='gasoline': dem_cols=['energy_demand_enfu_subsector_total_pj_trns_fuel_gasoline'], eff_cols=['fuelefficiency_trns_road_light_gasoline_km_per_litre']
Fuel='hydrocarbon_gas_liquids': dem_cols=['energy_demand_enfu_subsector_total_pj_trns_fuel_hydrocarbon_gas_liquids'], eff_cols=['fuelefficiency_trns_road_light_hydrocarbon_gas_liquids_km_per_litre']
Fuel='hydrogen': dem_cols=['energy_demand_enfu_subsector_total_pj_trns_fuel_hydrogen'], eff_cols=['fuelefficiency_trns_road_light_hydrogen_km_per_litre']
Fuel='natural_gas': dem_cols=['energy_demand_enfu_subsector_total_pj_trns_fuel_natural_gas'], eff_cols=[]
  → skipping 'natural_gas' (no matching columns)



In [11]:
# 6) Build output (with identifiers)
output_trns = pd.DataFrame({
    'primary_id': base_case['primary_id'],
    'region':     base_case['region'],
    'time_period': base_case['time_period'],
    'transportation_volume_saved_in_PJ': total_saved_volume
}, index=base_case.index)

# 7) Apply $880 000 per PJ
capex_multiplier_trns = 880_000
output_trns['transportation_efficiency_capex'] = (
    output_trns['transportation_volume_saved_in_PJ'] * capex_multiplier_trns
)
output_trns['transportation_efficiency_opex'] = 0



In [12]:
output_trns

,primary_id,region,time_period,transportation_volume_saved_in_PJ,transportation_efficiency_capex,transportation_efficiency_opex
0,0,louisiana,0,0.000000,0.000000e+00,0
1,0,louisiana,1,0.014345,1.262368e+04,0
2,0,louisiana,2,0.029470,2.593372e+04,0
3,0,louisiana,3,0.050429,4.437719e+04,0
4,0,louisiana,4,0.077275,6.800232e+04,0
5,0,louisiana,5,0.130341,1.146998e+05,0
6,0,louisiana,6,0.162717,1.431912e+05,0
7,0,louisiana,7,0.589068,5.183801e+05,0
8,0,louisiana,8,1.025977,9.028602e+05,0
9,0,louisiana,9,1.475988,1.298870e+06,0


In [13]:
output_trns = (
    output_trns
        .drop(columns='transportation_efficiency_opex')   # remove the two intermediate columns
        .rename(columns={'transportation_efficiency_capex': 'capex'})             # rename to capex
)

In [14]:
output_trns

,primary_id,region,time_period,transportation_volume_saved_in_PJ,capex
0,0,louisiana,0,0.000000,0.000000e+00
1,0,louisiana,1,0.014345,1.262368e+04
2,0,louisiana,2,0.029470,2.593372e+04
3,0,louisiana,3,0.050429,4.437719e+04
4,0,louisiana,4,0.077275,6.800232e+04
5,0,louisiana,5,0.130341,1.146998e+05
6,0,louisiana,6,0.162717,1.431912e+05
7,0,louisiana,7,0.589068,5.183801e+05
8,0,louisiana,8,1.025977,9.028602e+05
9,0,louisiana,9,1.475988,1.298870e+06


In [15]:
output_trns.to_csv(OUTPUT_DIR / "transportation_efficiency_all_fules.csv")